# Vectors

1. Import and define the embedding model

In [1]:
# Import the embedding Library
#  Define the Model

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embed the data and the question/query

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

# The Dot Product. 

Identifies the closeness of the value being searched vs the data

In [3]:
v1.dot(dv)

np.float32(0.3233239)

In [4]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

v2.dot(dv)

np.float32(0.019730505)

Loading the data

Generating embeddings by chunking the data by 50

In [5]:
import sys
import os
# Add the parent directory to the Python search path
sys.path.append(os.path.abspath(os.path.join('..')))

from ingest import load_faq_data

documents = load_faq_data()

for num, content in enumerate(documents):
    print(f"{num:} : {content}" )

Status code: 200
0 : {'id': '9e508f2212', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: When does the course start?', 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}
1 : {'id': 'bfafa427b3', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: What are the prerequisites for this course?', 'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (he

In [6]:
texts = []


# Since the raw data is in a dictionary structure, for accessibility compile them in as list of strings.
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

print(texts)

# for num, content in enumerate(texts):
#     print(f"\n{num:} : {content}" )

["Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.", "Course: What are the prerequisites for this course? To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up 

## RAG Vector Search: Batch Embedding Generation


What it does: Converts text into vectors (embeddings) using a model so they can be searched semantically.

Why batch (chunk) it?
Memory: Prevents system crashes (Out-of-Memory) by processing small, predictable chunks.

Speed: Maximizes GPU efficiency through parallel processing (faster than encoding one-by-one).

API Limits: Avoids payload size limits and rate-limit errors (429) when using external API services.




In [7]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [8]:
print(vectors[0])
vectors[0].shape

[-2.67062075e-02 -1.22457527e-01  1.59441940e-02  6.09419867e-03
 -1.11915218e-02 -6.55021369e-02 -7.62883723e-02 -2.08819825e-02
 -9.27567780e-02  3.55664939e-02  3.15623283e-02 -1.09017240e-02
 -2.45336164e-02 -1.82476491e-02  3.43917459e-02 -1.32052237e-02
  7.22679775e-03 -1.54126689e-01  6.41842633e-02 -9.07447562e-03
  3.94655354e-02 -2.99638007e-02  2.08512526e-02  3.71391885e-02
  3.52526829e-02 -2.49496219e-03  7.71129057e-02  2.78048366e-02
  1.54832313e-02  4.92820097e-03  1.48517417e-03  3.93927172e-02
  7.25188628e-02  9.02841091e-02  5.25653400e-02  2.72272509e-02
  3.86085436e-02 -7.50263184e-02 -2.48754360e-02  1.06018238e-01
 -4.82870899e-02 -5.00598289e-02 -4.16486450e-02  7.91618004e-02
  5.06461747e-02 -3.68654437e-04 -2.83320178e-03 -2.59598251e-02
 -3.82818282e-02  8.59545320e-02 -2.85154935e-02 -7.23346695e-02
 -2.05845684e-02  6.25361130e-02 -3.35789174e-02  9.84005332e-02
 -4.31055091e-02  8.77074674e-02 -5.19731455e-02  3.86910047e-03
 -3.51885036e-02 -8.52016

(384,)

## Searching a question similarity with the vectors

In [9]:
import numpy as np 

numpy_vectors = np.array(vectors)

query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [10]:
scores = numpy_vectors.dot(v_query)
print(scores)

[ 0.48740578  0.20991941  0.7629409  ... -0.08637967  0.03759795
 -0.03037035]


np.argmax() returns the index with the highest value

In [11]:
highest = np.argmax(scores) 
print(highest)
print(vectors[highest])
print("\nANSWER:\n",(documents[highest]["answer"]))

2
[-8.89655054e-02 -6.12817965e-02  7.75606232e-03 -1.48411840e-02
  5.76542020e-02  3.91595215e-02 -1.01738952e-01  5.06818993e-03
 -1.13026381e-01  8.81776027e-03  2.82028429e-02  2.14396454e-02
 -4.82826261e-03  4.61629517e-02  3.01450156e-02  3.44363637e-02
 -4.42705564e-02 -5.48175611e-02  5.62665537e-02 -1.81908794e-02
 -2.86035519e-02 -3.57326865e-02 -3.72983813e-02  3.58115360e-02
  6.05606213e-02  1.14685394e-01  7.00178966e-02  2.85892817e-03
  3.82801369e-02  6.65671565e-03 -6.63070902e-02  1.49549963e-02
  1.08069237e-02 -2.49716919e-02  9.01510194e-02 -1.19031789e-02
  5.90765513e-02 -2.70682611e-02 -2.43295804e-02 -2.43635662e-02
 -4.54950929e-02 -4.48936038e-02 -6.00351952e-02  6.58893511e-02
 -3.77624133e-03  1.72289442e-02  2.23840922e-02 -9.53663960e-02
 -5.73817194e-02  3.99230234e-02  6.39053732e-02  2.78499327e-03
 -1.00702740e-01 -3.12122889e-03 -2.04044618e-02  8.58407095e-02
  6.72884732e-02 -1.77011471e-02 -7.72469714e-02 -4.17660587e-02
 -2.96824686e-02 -3.116

## Python Slice Pattern:

[start:stop:step]

- If you leave something blank, Python fills in a default.
- The Negative indices (-n): means counting from the end of the sequence instead of the start.

In [12]:
top5 = np.argsort(scores)[-5:]
print(scores[top5][::-1])

[0.7629409  0.757937   0.7192131  0.6536312  0.56009996]


Workaround

In [13]:
highest_five = np.argsort(-scores)[:5] 
print(highest_five)

[  2 625 907 538   7]


printing each document using the top 5 indexes

In [14]:
for num, i in enumerate(highest_five):
    print(f"{num+1}: {documents[i]["answer"]}\n")

1: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

2: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

3: Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.

In order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.

4: Yes, but if you want to receive a certificate, you need

## Vector Search with MinSearch

1. Initialization of the Index: idefines the structure of the index: which fields are searchable (text) and which are filterable (keywords).

2.  Embedding : processes the documents, generates vector representations, and builds the internal search structures that allows for efficient searching.


In [15]:
from minsearch import VectorSearch


# Define the Index Value 
vindex = VectorSearch(keyword_fields=["course"])

# Pass the embeddings and Documents/Data
vindex.fit(numpy_vectors, documents)

In [16]:
result = vindex.search(v_query, 
                        num_results=5, 
                        filter_dict={'course':'llm-zoomcamp'})

print(result[0], "\n")

result_texts = []

for context in result:
    text = "Question: " + context['question'] + "\n" + context['answer'] + "\n"
    result_texts.append(text)

for conext in result_texts:
    print(conext)

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'} 

Question: I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Question: When will the course be offered next?
Summer 2027.

Question: Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

Question: Can I run the

In [17]:
import os
from dotenv import load_dotenv
from google.genai import Client
# from rag_helper import RAGBase

# Load the .env file
load_dotenv()
print(f"Key found: {os.environ.get('GEMINI_API_KEY') is not None}")

gemini_api_client = Client(api_key=os.environ.get('GEMINI_API_KEY'))
print(f"Client created: {gemini_api_client is not None}")


Key found: True
Client created: True


In [18]:
from rag_vector_helper import RAGVector

vector_assistant = RAGVector(
                    embedder=model,
                    index=vindex,
                    llm_client=gemini_api_client
)

In [19]:
# vector_assistant.ask("I just found out about the program, can I still sign up?")

# Search with  SQLite

Note: the mode parameter controls which approximate nearest neighbor (ANN) algorithm is used for vector search

In [20]:
from sqlitesearch import VectorSearchIndex

# Creates an index and db where the vectors will be stored

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",  #
    db_path="faq_vectors2.db"
)

vs_index.fit(numpy_vectors, documents)
